# Cleaning Downloaded Data from avian-flu

Author: Alexander Maksiaev

Purpose: Clean downloaded data from avian-flu, rename sequences according to convention, de-duplicate from GISAID

In [ ]:
# Housekeeping

import os
import glob 
import pandas as pd
import xml.etree.ElementTree as ET
import requests
import time
import numpy as np
import importlib
import utils  
importlib.reload(utils)
from utils import * 

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu"
# downloads = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
downloads = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
originals = downloads + "Andersen_Downloads/"
temp_files = downloads + "Andersen_Temp_Files/"
complete_files = downloads + "Andersen_Complete_Files/"

os.chdir(downloads)

In [ ]:
# Read metadata

metadata_folder = originals + "avian-influenza/metadata/"
os.chdir(metadata_folder)

metadata = pd.read_csv("SraRunTable_automated.csv")

print(len(metadata)) # 7397 rows

# Split date format to only check year
for date in metadata["Collection_Date"]:
    if "/" in date or date == "missing":
        metadata = metadata[metadata["Collection_Date"] != date]
metadata["Collection_Date"] = metadata["Collection_Date"].apply(lambda x: x.split("-")[0])
metadata["Collection_Date"] = metadata["Collection_Date"].apply(lambda x: int(x))

# Find only >= 2024 using run ID from metadata
metadata_new = metadata[metadata["Collection_Date"] >= 2024]
metadata_new["Collection_Date"] = metadata_new["Collection_Date"].astype(int) # Years are not floats

print(len(metadata_new)) # 6053 rows

In [ ]:
# Modify isolates to xxxxxx-xxx

# new_isolates = []
# for isolate in metadata_new["isolate"]:
#     new_isolate = ""
#     # if isolate:
#     #     print(isolate)
#     try:
#         split_isolate = isolate.split("-")
#     except:
#         new_isolate = ""
#     # else:
#     #     split_isolate = ""
#     for split in split_isolate:
#         if len(split) == 6:
#             new_isolate = new_isolate + split
#         if len(split) == 3:
#             new_isolate = new_isolate + "-" + split
#     new_isolates.append(new_isolate)

# metadata_new["isolate"] = new_isolates

# display(metadata_new["isolate"])

### Naming convention ###
>A/[host]/[geo_loc_name]/[isolate]/[year]|[serotype: H5N1]|[collection_date]|[host_type]|[genotype: B3.13 or D1.1]

host_type is from manual animal reference

In metadata, we have: host, geo_loc_name, isolate, year

We need: geo_loc_name, collection_date, host_type, genotype

host = Host

geo_loc_name (primary) = geo_loc_name

geo_loc_name (secondary) = genbank_mapping.tsv > genbank_name

isolate = isolate

collection date (primary) = Collection_Date

collection date (secondary) = https://www.ncbi.nlm.nih.gov/genbank/ > BioSample (input: BioSample) > Nucleotide > [first result] > collection_date

serotype = serotype

host type = [from ref] 

genotype = [from genoflu] -- use output.tsv

In [ ]:
# Get genotype from genoflu
os.chdir(temp_files)
output_tsv = pd.read_csv("output.tsv", delimiter="\t")

b313_and_d11_only = output_tsv[(output_tsv["Genotype"] == "B3.13") | (output_tsv["Genotype"] == "D1.1")]
b313_and_d11_only = b313_and_d11_only.rename(columns={"sample": "Run"})
b313_and_d11_only = b313_and_d11_only.drop_duplicates(subset="Run", keep="last")
# print(b313_and_d11_only)
print(len(b313_and_d11_only)) # 5160 rows

metadata_new = metadata_new.merge(b313_and_d11_only, on="Run", how="inner")

print(len(metadata_new)) # 5160

In [ ]:
# Get animals from animal reference
os.chdir(downloads)
animals_ref = pd.read_csv("animals_ref.csv")
fix_animals_andersen(metadata_new, animals_ref) # Get host type
metadata_new["years"] = metadata_new["Collection_Date"].apply(lambda x: str(x).split("-")[0]) # Get year only from collection date

In [ ]:
print(len(metadata_new))

In [ ]:
# Get geolocation from genbank_mapping.tsv

os.chdir(metadata_folder)
genbank_mapping = pd.read_csv("genbank_mapping.tsv", delimiter="\t")
genbank_mapping["Run"] = genbank_mapping["sra_run"]
genbank_mapping = genbank_mapping.drop_duplicates(subset="Run")
genbank_mapping["name_state"] = genbank_mapping["genbank_name"].apply(lambda x: x.split("/")[2])
# genbank_mapping["isolate"] = genbank_mapping["genbank_name"].apply(lambda x: x.split("/")[3])

# Change isolate names in genbank mapping

# new_isolates = []
# for isolate in genbank_mapping["isolate"]:
#     new_isolate = ""
#     # if isolate:
#     #     print(isolate)
#     try:
#         split_isolate = isolate.split("-")
#     except:
#         continue
#     # else:
#     #     split_isolate = ""
#     for split in split_isolate:
#         if len(split) == 6:
#             new_isolate = new_isolate + split
#         if len(split) == 3:
#             new_isolate = new_isolate + "-" + split
#     new_isolates.append(new_isolate)

# genbank_mapping["isolate"] = new_isolates


# map to metadata


metadata_genbank = metadata_new.merge(genbank_mapping, on="Run", how="inner")

print(len(metadata_genbank))
display(metadata_genbank)

In [ ]:
# Get collection date from GenBank eutils 

def search_collection_date(biosample):

    print(biosample)

    try:

        # Avoid spamming the server
        time.sleep(2)
    
        base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
        search_url = base_url + "esearch.fcgi?db=biosample&term=" + biosample +"&usehistory=y&api_key=2cbaf77ac9ec5ae7844ea350076ae6d56809"

        # Get Biosample ID from search_url
        output = requests.get(search_url)
        xml = output.content
        root = ET.fromstring(xml)
        sample_id = root.find("./IdList/Id").text

        biosample_url = base_url + "elink.fcgi?dbfrom=biosample&db=nuccore&id=" + sample_id + "&cmd=neighbor_history&api_key=2cbaf77ac9ec5ae7844ea350076ae6d56809"
        
        # Get Nucleotide ID from biosample_url
        output = requests.get(biosample_url)
        xml = output.content
        root = ET.fromstring(xml)
        query_key = root.find(".//QueryKey").text
        web_env = root.find(".//WebEnv").text

        nucleotide_url = base_url + "esummary.fcgi?db=nuccore&query_key=" + query_key + "&WebEnv=" + web_env + "&version=2.0&api_key=2cbaf77ac9ec5ae7844ea350076ae6d56809"

        output = requests.get(nucleotide_url) 
        xml = output.content
        root = ET.fromstring(xml)

        # Grab collection date at the end of the sub name
        collection_date = root.find(".//SubName").text.split("|")[-1]

        return collection_date
    
    except:
        print("Unable to find collection date.")

        if len(metadata_genbank[metadata_genbank["BioSample"] == biosample]["years"]) > 0: # If a year exists
            collection_date = metadata_genbank[metadata_genbank["BioSample"] == biosample]["years"].values[0]
        else:
            collection_date = float('nan') 

        return collection_date
    
metadata_genbank["Collection_Date_Specific"] = metadata_genbank["BioSample"].apply(search_collection_date)

In [ ]:
# Save this so we don't have to do it again

os.chdir(temp_files)
metadata_genbank.to_csv("metadata_genbank.csv")

In [ ]:
# Make names

# names = ">A/" + metadata_new["Host"] + "/" + metadata_new["geo_loc_name"] + "/" + metadata_new["isolate"] + "/" + years + "|H5N1|" + metadata_new["Collection_Date"].apply(lambda x: str(x)) + "|" + metadata_new["Host_Type"] + "|" + metadata_new["Genotype"]
names = ">A/" + metadata_genbank["Host"] + "/" + metadata_genbank["name_state"] + "/" + metadata_genbank["isolate"] + "/" + metadata_genbank["years"] + "|H5N1|" + metadata_genbank["Collection_Date_Specific"] + "|" + metadata_genbank["Host_Type"] + "|" + metadata_genbank["Genotype"]

metadata_genbank["Name"] = names

display(metadata_genbank)

In [ ]:
# Make fasta files

fasta_folder = originals + "avian-influenza/fasta/"

os.chdir(fasta_folder)

pairs = []
fasta_files = {}

for genotype in ["B3.13", "D1.1"]:
    for segment in ["PB2", "PB1", "PA", "NS", "NP", "NA", "MP", "HA"]:
        pair = genotype + "_" + segment
        pairs.append(pair)

for pair in pairs:
    fasta_files[pair] = [] # List to hold fasta files

for run in metadata_genbank["Run"].values: # For each run 
    for dirpath, dirs, files in os.walk(fasta_folder): # Find the fasta file
        for file in files:
            file_name = os.path.join(dirpath, file) # Get file name
            # print(file_name)
            if run in file_name: # Note that there will be ~8 files total with that run name
                # Make a fasta file and put it in the list
                with open(file_name) as f:
                    lines = f.readlines()
                    sequence = lines[1] 
                    # Each run/segment pair has one sequence -- it's placed into a file with other run/segment pairs with the same segment and genotype
                    header = metadata_genbank[metadata_genbank["Run"] == run].loc[:, "Name"].values[0]
                    genotype = metadata_genbank[metadata_genbank["Run"] == run].loc[:, "Genotype"].values[0]
                    # print(header)
                    # print(genotype)
                    # break 
                    segment = file_name.split("_")[-2]
                    # Find the pair that corresponds to 
                    pair_name = genotype + "_" + segment
                    this_specific_fasta = []
                    for pair in pairs:
                        # print(pair)
                        # print(pair_name)
                        if pair_name == pair:
                            this_specific_fasta.append(header)
                            this_specific_fasta.append(sequence)
                            fasta_files[pair].append(this_specific_fasta)
                f.close()

In [ ]:
# print(fasta_files[list(fasta_files.keys())[0]])

print(len(fasta_files["B3.13_PB1"]))

print(fasta_files["B3.13_PB1"])

In [ ]:
# Create fasta files 
os.chdir(complete_files)
for pair in fasta_files.keys():
    output_path = complete_files + pair + ".fasta" 

    output_file = open(output_path, "w")
    for item in fasta_files[pair]:
        # for item in item:
        # item = fasta_files[pair]
        try:
            name = str(item[0].values[0]) # See if this is one we didn't have a collection date for
        except:
            name = str(item[0])
        print(name)
        # First is header, second is sequence
        # print(value)
        output_file.write(name + "\n")
        output_file.write(item[1])
    output_file.close()

In [41]:
# De-duplication with GISAID

gisaid = downloads + "GISAID_Complete_Fasta_Files/"

os.chdir(gisaid)

isolates = {}
for dirpath, dirs, files in os.walk(gisaid): # Find the fasta file
    for file in files:
        file_name = os.path.join(dirpath, file) # Get file name
        with open(file_name) as f:
            lines = f.readlines()
            # Some lines start with 25_, others 25-. This shouldn't matter, but split on "_" first
            for num, line in enumerate(lines):
                if line[0] == ">": # If it's a header
                    full = line.split("/")[3] # Get the isolate
                    partial = full.split("_")[-1] # If 25_, get the last bit
                    digits = partial.split("-")
                    isolate = ""
                    other = ""
                    for d in digits:
                        # print(d)
                        if len(d) == 6 and d.isnumeric(): # If it's just digits and not one of those weird isolates
                            isolate = d + "-"
                        elif len(d) == 3 and d.isnumeric():
                            isolate = isolate + d
                        elif d.isnumeric() == False: # If it's a weird isolate
                            other = d + "-"
                        else: 
                            other = other + d
                    # Now add to list to check in Andersen files without doing wild for loops
                    if len(isolate) == 10: # If this is a correctly formatted isolate
                        # isolates.append(isolate)
                        # All headers are followed by sequences
                        isolates[isolate] = [line, lines[num + 1]]
                    else: # If this is some other isolate
                        isolates[other] = [line, lines[num + 1]]

# If this isolate is not already in the Andersen fasta files, add it to those files

os.chdir(complete_files)

andersen_isolates = []

# Some names don't exist
metadata_genbank = metadata_genbank.dropna()
# Exclude duplicates
metadata_genbank = metadata_genbank.drop_duplicates()

for name in metadata_genbank["Name"]:
    # print(name)
    # if :
    full = name.split("/")[3] # Get the isolate
    partial = full.split("_")[-1] # If 25_, get the last bit
    digits = partial.split("-")
    isolate = ""
    other = ""
    for d in digits:
        # print(d)
        if len(d) == 6 and d.isnumeric(): # If it's just digits and not one of those weird isolates
            isolate = d + "-"
        elif len(d) == 3 and d.isnumeric():
            isolate = isolate + d
        elif d.isnumeric() == False: # If it's a weird isolate
            other = d + "-"
        else: 
            other = other + d
    if len(isolate) > 0:
        andersen_isolates.append(isolate)
    elif len(other) > 0:
        andersen_isolates.append(other)

# Check if the isolate (or other) is in the keys of the dictionary made above
for key in isolates.keys():
    for isolate in andersen_isolates:
        if isolate == key: # We've seen it
            # print(isolate)
            isolates.keys().remove(isolate) # Remove it from the list

# Now find in original GISAID data and add to metadata_genbank
gisaid_data = pd.DataFrame()

display(metadata_genbank)

,Run,Assay Type,AvgSpotLen,Bases,BioProject,BioSample,BioSampleModel,Bytes,Center Name,Collection_Date,...,seg_file,seg_seq_name,sra_run,seg,genbank_acc,genbank_seg,genbank_name,name_state,Collection_Date_Specific,Name
1130,SRR30332676,WGS,146.43,134279049,PRJNA1102327,SAMN43274409,Viral,47241506,USDA-NVSL,2024,...,SRR30332676_HA_cns.fa,Consensus_SRR30332676_HA_cns_threshold_0.5_qua...,SRR30332676,HA,PQ379799.1,4,A/chicken/CO/24-020796-008-original/2024,CO,2024,>A/Chicken/CO/24-020796-008/2024|H5N1|2024|avi...
1131,SRR30332677,WGS,146.72,184920982,PRJNA1102327,SAMN43274408,Viral,64922333,USDA-NVSL,2024,...,SRR30332677_HA_cns.fa,Consensus_SRR30332677_HA_cns_threshold_0.5_qua...,SRR30332677,HA,PQ379791.1,4,A/chicken/CO/24-020796-007-original/2024,CO,2024,>A/Chicken/CO/24-020796-007/2024|H5N1|2024|avi...
1132,SRR30332678,WGS,147.04,135374111,PRJNA1102327,SAMN43274407,Viral,47883641,USDA-NVSL,2024,...,SRR30332678_HA_cns.fa,Consensus_SRR30332678_HA_cns_threshold_0.5_qua...,SRR30332678,HA,PQ379783.1,4,A/chicken/CO/24-020796-006-original/2024,CO,2024,>A/Chicken/CO/24-020796-006/2024|H5N1|2024|avi...
1133,SRR30332679,WGS,144.88,1040492099,PRJNA1102327,SAMN43274466,Viral,365588574,USDA-NVSL,2024,...,SRR30332679_HA_cns.fa,Consensus_SRR30332679_HA_cns_threshold_0.5_qua...,SRR30332679,HA,PQ423386.1,4,A/cattle/CO/24-022179-001-original/2024,CO,2024,>A/Cattle/CO/24-022179-001/2024|H5N1|2024|catt...
1134,SRR30332680,WGS,146.06,206327767,PRJNA1102327,SAMN43274465,Viral,72336073,USDA-NVSL,2024,...,SRR30332680_HA_cns.fa,Consensus_SRR30332680_HA_cns_threshold_0.5_qua...,SRR30332680,HA,PQ423378.1,4,A/cattle/CO/24-022177-001-original/2024,CO,2024,>A/Cattle/CO/24-022177-001/2024|H5N1|2024|catt...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2085,SRR31597262,WGS,147.82,70922269,PRJNA1102327,SAMN45128860,Viral,25383070,USDA-NVSL,2024,...,SRR31597262_HA_cns.fa,Consensus_SRR31597262_HA_cns_threshold_0.5_qua...,SRR31597262,HA,PQ797665.1,4,A/chicken/CA/24-031666-015-original/2024,CA,2024,>A/Chicken/CA/24-031666-015/2024|H5N1|2024|avi...
2086,SRR31597263,WGS,148.29,81598872,PRJNA1102327,SAMN45128859,Viral,29223378,USDA-NVSL,2024,...,SRR31597263_HA_cns.fa,Consensus_SRR31597263_HA_cns_threshold_0.5_qua...,SRR31597263,HA,PQ797657.1,4,A/chicken/CA/24-031666-014-original/2024,CA,2024,>A/Chicken/CA/24-031666-014/2024|H5N1|2024|avi...
2087,SRR31597264,WGS,147.42,86023852,PRJNA1102327,SAMN45128772,Viral,30874643,USDA-NVSL,2024,...,SRR31597264_HA_cns.fa,Consensus_SRR31597264_HA_cns_threshold_0.5_qua...,SRR31597264,HA,PQ797249.1,4,A/Turkey/CA/24-031284-012-original/2024,CA,2024,>A/Turkey/CA/24-031284-012/2024|H5N1|2024|avia...
2088,SRR31597265,WGS,147.43,84451161,PRJNA1102327,SAMN45128763,Viral,30317458,USDA-NVSL,2024,...,SRR31597265_HA_cns.fa,Consensus_SRR31597265_HA_cns_threshold_0.5_qua...,SRR31597265,HA,PQ797177.1,4,A/Turkey/CA/24-031284-002-original/2024,CA,2024,>A/Turkey/CA/24-031284-002/2024|H5N1|2024|avia...


In [ ]:
# def search_collection_date(biosample):
    
#     base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"
#     search_url = base_url + "esearch.fcgi?db=biosample&term=" + biosample +"&usehistory=y&api_key=2cbaf77ac9ec5ae7844ea350076ae6d56809"

#     # Get Biosample ID from search_url
#     output = requests.get(search_url)
#     xml = output.content
#     root = ET.fromstring(xml)
#     sample_id = root.find("./IdList/Id").text

#     biosample_url = base_url + "elink.fcgi?dbfrom=biosample&db=nuccore&id=" + sample_id + "&cmd=neighbor_history&api_key=2cbaf77ac9ec5ae7844ea350076ae6d56809"
    
#     # Get Nucleotide ID from biosample_url
#     output = requests.get(biosample_url)
#     xml = output.content
#     root = ET.fromstring(xml)
#     query_key = root.find(".//QueryKey").text
#     web_env = root.find(".//WebEnv").text

#     nucleotide_url = base_url + "esummary.fcgi?db=nuccore&query_key=" + query_key + "&WebEnv=" + web_env + "&version=2.0&api_key=2cbaf77ac9ec5ae7844ea350076ae6d56809"

#     output = requests.get(nucleotide_url) 
#     xml = output.content
#     root = ET.fromstring(xml)

#     # Grab collection date at the end of the sub name
#     collection_date = root.find(".//SubName").text.split("|")[-1]

#     # Avoid spamming the server
#     time.sleep(3)
    
#     return collection_date


In [ ]:
# example = search_collection_date("SAMN41019216")
# print(example)



In [ ]:

# unique_animals_all = sort_animals_anderson(metadata_new)

# # Flatten unique_animals
# every_unique_animal = []
# for animal in unique_animals_all:
#     every_unique_animal.append(animal)

# print(every_unique_animal)

# unique_animals_set = list(set(every_unique_animal))
# # animals_df = pd.DataFrame(columns=["avian", "cattle", "feline", "other_mammal", "human", "other"])
# # animals_df["other"] = unique_animals_set # to sort

# os.chdir(downloads)

# animals_ref = pd.read_csv("animals_ref.csv")


# # If animal not in ref1, put in ref2

# common_animals = []
# # Check if animals in unique_animals_set are in ref1
# for animal in unique_animals_set:
#     for col in animals_ref.columns:
#         if animal in animals_ref[col].values and type(animal) == str:
#             common_animals.append(animal)

# print(common_animals)
# print(len(common_animals))

# different_animals = []
# for animal in unique_animals_set:
#     if animal not in common_animals:
#         different_animals.append(animal)

# print(different_animals)

# # Add to dataframe
# animals_df = animals_ref
# # Make different_animals same length as dataframe, if shorter
# if len(different_animals) < len(animals_df):
#     number_of_times_to_add_nan = len(animals_df) - len(different_animals)
#     for i in range(number_of_times_to_add_nan):
#         different_animals.append(float('nan'))
# # If longer, deal with that later

# animals_df["new"] = (different_animals)

# print(animals_df)

# animals_df.to_csv("animals_ref_to_sort.csv")
